# CellIQ CT nodule detector: evaluation on public LUNA16 scans

This notebook runs the open-source MONAI `lung_nodule_ct_detection` bundle (Apache-2.0) on randomly chosen LUNA16 `subset0` scans
and measures **how many radiologist-marked nodules it finds against how many false alarms per scan** (an FROC analysis),
with a bootstrap confidence interval.

It produces `cellIQ_ct_eval.zip` containing `eval_summary.json` and `froc.png`, which the CellIQ app reads from `demo_ct/eval/`.

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. Takes about 15 to 25 minutes.

**What a result means:** a detection is a nodule *candidate* for a radiologist to look at. It is not a cancer diagnosis.

Credit: LUNA16 (https://luna16.grand-challenge.org, CC BY 4.0), built from LIDC-IDRI (CC BY 3.0). Research use only.

## 1. Settings (the only cell you normally edit)

In [ ]:
import os, sys, json, time, subprocess, shutil

N_SCANS = 24          # how many scans to test
MAX_RAW_MB = 120      # only scans whose compressed raw file is at most this big (keeps downloads quick)
SEED = 0              # scan selection seed (same seed = same scans)
BOOT_N = 500          # bootstrap repeats for the confidence interval
BOOT_SEED = 1
FETCH_THREADS = 4

BUNDLE = "/content/bundles/lung_nodule_ct_detection"
LUNA_DIR = "/content/luna"
CASE_DIR = "/content/cases"
DET_DIR = "/content/det_out"
OUT_DIR = "/content/eval_out"
ZENODO_RECORDS = [3723295, 2595812, 2595813, 2596478, 4121926]   # LUNA16 records on Zenodo


def run(cmd, **kw):
    """Run a command, return the finished process, and show the tail of its output if it failed."""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode:
        print("COMMAND FAILED:", " ".join(map(str, cmd))[:200])
        print((r.stdout + r.stderr)[-1500:])
    return r


for d in (LUNA_DIR, CASE_DIR, DET_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print("settings ok")

## 2. Install the libraries and download the detector bundle

In [ ]:
r = run([sys.executable, "-m", "pip", "install", "-q", "monai[itk,nibabel,tqdm]", "fire", "pytorch-ignite",
         "SimpleITK", "pandas", "remotezip", "matplotlib"])
assert r.returncode == 0, "pip install failed (see above)"
import monai, torch
print("monai", monai.__version__, "| GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Switch the runtime to T4 GPU or inference will be very slow.")

r = run([sys.executable, "-m", "monai.bundle", "download", "lung_nodule_ct_detection", "--bundle_dir", "/content/bundles"])
assert r.returncode == 0 and os.path.isdir(BUNDLE), "bundle download failed (see above)"
print("bundle ready:", BUNDLE)

## 3. Find the LUNA16 files on Zenodo and download the annotation tables

In [ ]:
import requests

WANT = {"subset0.zip": None, "annotations.csv": None, "annotations_excluded.csv": None}
for rid in ZENODO_RECORDS:
    try:
        rec = requests.get(f"https://zenodo.org/api/records/{rid}", timeout=30).json()
    except Exception as e:
        print("record", rid, "lookup failed:", type(e).__name__)
        continue
    for f in rec.get("files", []):
        key = f.get("key") or f.get("filename")
        links = f.get("links") or {}
        if key in WANT and WANT[key] is None:
            WANT[key] = {"record": rid, "url": links.get("self") or links.get("download"), "size": f.get("size")}
print({k: (v["record"], v["size"]) if v else None for k, v in WANT.items()})
assert WANT["subset0.zip"] and WANT["annotations.csv"], "subset0.zip or annotations.csv was not found on Zenodo"

for key in ("annotations.csv", "annotations_excluded.csv"):
    if WANT[key]:
        run(["curl", "-L", "-s", "--retry", "3", "-o", f"{LUNA_DIR}/{key}", WANT[key]["url"]])
        print(key, os.path.getsize(f"{LUNA_DIR}/{key}"), "bytes")
SUBSET_URL = WANT["subset0.zip"]["url"]

## 4. Pick scans at random and fetch only those files from the remote zip

In [ ]:
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from remotezip import RemoteZip

ann = pd.read_csv(f"{LUNA_DIR}/annotations.csv")
excl_path = f"{LUNA_DIR}/annotations_excluded.csv"
excl = pd.read_csv(excl_path) if os.path.exists(excl_path) else None

with RemoteZip(SUBSET_URL) as rz:
    infos = {i.filename: i for i in rz.infolist()}
mhd = sorted(n for n in infos if n.endswith(".mhd"))
eligible = [n for n in mhd if n[:-4] + ".raw" in infos and infos[n[:-4] + ".raw"].compress_size <= MAX_RAW_MB * 1e6]
rng = np.random.RandomState(SEED)
chosen = sorted(rng.choice(eligible, size=min(N_SCANS, len(eligible)), replace=False).tolist())
print(f"scans in subset0: {len(mhd)} | eligible: {len(eligible)} | chosen: {len(chosen)}")
print("compressed data to fetch: %.0f MB" % (sum(infos[n[:-4] + ".raw"].compress_size for n in chosen) / 1e6))


def fetch(name):
    for attempt in range(3):
        try:
            with RemoteZip(SUBSET_URL) as z:
                z.extract(name, CASE_DIR)
                z.extract(name[:-4] + ".raw", CASE_DIR)
            return name, True
        except Exception:
            time.sleep(3)
    return name, False


t0, done = time.time(), 0
with ThreadPoolExecutor(max_workers=FETCH_THREADS) as ex:
    for fut in as_completed([ex.submit(fetch, n) for n in chosen]):
        name, ok = fut.result()
        done += 1
        print(f"{done}/{len(chosen)} {'ok' if ok else 'FAILED'} {os.path.basename(name)[-12:-4]} | {time.time() - t0:.0f}s", flush=True)

scans = [n for n in chosen if os.path.exists(f"{CASE_DIR}/{n[:-4]}.raw") and os.path.exists(f"{CASE_DIR}/{n}")]
assert scans, "no scans were downloaded"
uids = [os.path.basename(n)[:-4] for n in scans]
json.dump({"validation": [{"image": n} for n in scans]}, open("/content/datalist.json", "w"))
print("ready:", len(scans), "scans | with marked nodules:", sum(u in set(ann.seriesuid) for u in uids),
      "| marked nodules:", int(ann.seriesuid.isin(uids).sum()))

## 5. Run the detector on every chosen scan

In [ ]:
cfg = json.load(open(f"{BUNDLE}/configs/inference.json"))
cfg["whether_raw_luna16"] = True                 # the scans are raw LUNA16 volumes
cfg["dataset_dir"] = CASE_DIR
cfg["data_list_file_path"] = "/content/datalist.json"
cfg["output_dir"] = DET_DIR
cfg["output_filename"] = "result.json"
cfg["dataloader"]["num_workers"] = 0             # more workers ran out of RAM on Colab
json.dump(cfg, open("/content/infer_cfg.json", "w"), indent=1)

t0 = time.time()
r = run([sys.executable, "-m", "monai.bundle", "run", "--config_file", "/content/infer_cfg.json",
         "--meta_file", f"{BUNDLE}/configs/metadata.json"], cwd=BUNDLE)
print(f"inference exit code: {r.returncode} | {time.time() - t0:.0f}s")
assert r.returncode == 0, "inference failed (see above)"

raw = json.load(open(f"{DET_DIR}/result.json"))
records = raw if isinstance(raw, list) else raw.get("validation", [raw])
assert len(records) == len(scans), f"expected {len(scans)} results, got {len(records)}"
print("results:", len(records), "for", len(scans), "scans")

## 6. Match detections to the radiologists' marks

LUNA16 rule: a marked nodule counts as **found** when a detection's centre lies within that nodule's radius.
A detection that matches nothing is a **false alarm**, unless it lies within 5 mm of an "excluded" finding
(for example a very small nodule the reference list chose to ignore), in which case it is dropped.

In [ ]:
def record_matches(rec, uid):
    """True unless the record names a different scan (records without a file name are matched by order)."""
    img = str(rec.get("image", "")) if isinstance(rec, dict) else ""
    return (not img) or (uid in img)


assert all(record_matches(rec, uids[i]) for i, rec in enumerate(records)), "result order does not match the scan list"

per_scan = []
for i, rec in enumerate(records):
    uid = uids[i]
    boxes = np.array(rec["box"], dtype=float).reshape(-1, 6)           # [cx, cy, cz, w, h, d] in LUNA16 world mm
    scores = np.array(rec["label_scores"], dtype=float).reshape(-1)
    centres = boxes[:, :3]                                              # already world coordinates: no flip needed
    a = ann[ann.seriesuid == uid]
    marks = a[["coordX", "coordY", "coordZ"]].values
    radii = a["diameter_mm"].values / 2.0
    ignorable = excl[excl.seriesuid == uid][["coordX", "coordY", "coordZ"]].values if excl is not None else np.zeros((0, 3))

    best = np.zeros(len(marks))              # highest score of any detection that hits each marked nodule
    false_scores = []
    for c, s in zip(centres, scores):
        hit = np.linalg.norm(marks - c, axis=1) <= radii if len(marks) else np.zeros(0, bool)
        if hit.any():
            best[hit] = np.maximum(best[hit], s)
        elif len(ignorable) and (np.linalg.norm(ignorable - c, axis=1) <= 5.0).any():
            continue
        else:
            false_scores.append(s)
    per_scan.append({"uid": uid, "marked": len(marks), "best": best, "false": np.array(false_scores)})

n_scans = len(per_scan)
n_marked = int(sum(p["marked"] for p in per_scan))
n_with = int(sum(p["marked"] > 0 for p in per_scan))
print(f"scans: {n_scans} | scans with marked nodules: {n_with} | marked nodules: {n_marked}")

## 7. FROC numbers, confidence interval and score-threshold table

In [ ]:
THRESHOLDS = np.round(np.linspace(0.02, 0.999, 120), 4)
TARGETS = [0.125, 0.25, 0.5, 1, 2, 4, 8]       # false alarms per scan used for the table and the CPM


def froc(scan_ids):
    """Sensitivity and false alarms per scan at every score threshold, for the given (possibly repeated) scans."""
    n = len(scan_ids)
    best = np.concatenate([per_scan[i]["best"] for i in scan_ids]) if n else np.zeros(0)
    false = np.concatenate([per_scan[i]["false"] for i in scan_ids]) if n else np.zeros(0)
    sens = np.array([(best >= t).mean() if len(best) else 0.0 for t in THRESHOLDS])
    fpps = np.array([(false >= t).sum() / max(n, 1) for t in THRESHOLDS])
    return sens, fpps


def sens_at(sens, fpps, target):
    """Best sensitivity reachable without exceeding `target` false alarms per scan."""
    ok = sens[fpps <= target]
    return float(ok.max()) if len(ok) else 0.0


sens, fpps = froc(list(range(n_scans)))
table = {str(t): round(sens_at(sens, fpps, t), 3) for t in TARGETS}
cpm = float(np.mean([sens_at(sens, fpps, t) for t in TARGETS]))
print("share of marked nodules found, at most this many false alarms per scan:", table)
print("CPM (average over 1/8 to 8 false alarms per scan): %.3f" % cpm)

# Bootstrap over scans: resample whole scans with replacement and recompute
brng = np.random.RandomState(BOOT_SEED)
boots = {t: [] for t in (0.5, 1, 2)}
for _ in range(BOOT_N):
    ids = brng.randint(0, n_scans, n_scans)
    s_, f_ = froc(ids)
    for t in boots:
        boots[t].append(sens_at(s_, f_, t))
interval = {str(t): [round(float(np.percentile(v, 2.5)), 3), round(float(np.percentile(v, 97.5)), 3)] for t, v in boots.items()}
print("95% bootstrap range at 0.5 / 1 / 2 false alarms per scan:", interval)

by_score = {}
for t in (0.1, 0.3, 0.5, 0.7, 0.9):
    i = int(np.argmin(np.abs(THRESHOLDS - t)))
    by_score[str(t)] = {"sensitivity": round(float(sens[i]), 3), "false_alarms_per_scan": round(float(fpps[i]), 2)}
print("keeping only detections with score >= t:", by_score)

## 8. Draw the curve, save the results and make the download zip

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

order = np.argsort(fpps, kind="stable")
x, y = fpps[order], sens[order]
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.step(x, y, where="post", lw=2, label="detector")                      # step curve: sensitivity only changes at real detections
ax.plot(TARGETS, [sens_at(sens, fpps, t) for t in TARGETS], "o", ms=5, label="table points")
ax.set_xscale("log")
ax.set_xlim(0.1, 10)
ax.set_ylim(0, 1.02)
ax.set_xlabel("False alarms per scan")
ax.set_ylabel("Share of marked nodules found")
ax.set_title(f"FROC: {n_scans} LUNA16 scans, {n_marked} marked nodules")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/froc.png", dpi=140)
plt.close(fig)

summary = {
    "model": "MONAI lung_nodule_ct_detection v0.6.9 (Apache-2.0)",
    "data": "LUNA16 subset0 (CC BY 4.0) / LIDC-IDRI (CC BY 3.0)",
    "n_scans": n_scans, "n_scans_with_nodules": n_with, "n_marked_nodules": n_marked,
    "selection": f"random sample (seed {SEED}) of subset0 scans whose raw file is at most {MAX_RAW_MB} MB compressed (to keep downloads fast)",
    "rule": "a marked nodule counts as found if a detection's centre lies within the nodule's radius (LUNA16 rule); detections matching annotations_excluded findings are ignored",
    "sensitivity_at_false_alarms_per_scan": table,
    "cpm": round(cpm, 3),
    "bootstrap_95_interval": interval,
    "by_score_threshold": by_score,
    "froc": {"false_alarms_per_scan": [round(float(v), 3) for v in x], "sensitivity": [round(float(v), 3) for v in y]},
    "note": "Subset0 is believed (not verified) to be held out from the released model's training. Small sample: read the intervals.",
}
json.dump(summary, open(f"{OUT_DIR}/eval_summary.json", "w"), indent=1)
shutil.make_archive("/content/cellIQ_ct_eval", "zip", OUT_DIR)
print("zip ready:", os.path.getsize("/content/cellIQ_ct_eval.zip"), "bytes")
print(json.dumps({k: v for k, v in summary.items() if k not in ("froc", "selection", "rule", "note")}, indent=1)[:1800])

from IPython.display import Image, display
display(Image(f"{OUT_DIR}/froc.png"))
print("Download /content/cellIQ_ct_eval.zip from the file panel on the left.")

## 9. Scan-level check: does it flag the scans that have marked nodules?

A simpler question than the FROC curve: call a scan "flagged" if the detector produced at least one detection at or above a score,
and compare with whether the radiologists marked a nodule in that scan.
Only a few scans have no marked nodule, so treat the specificity as very rough. Detections near an "excluded" finding are not counted here.

In [ ]:
mx = np.array([max([*p["best"].tolist(), *p["false"].tolist(), 0.0]) for p in per_scan])   # highest detection score in each scan
has_nodule = np.array([p["marked"] > 0 for p in per_scan])
scan_level = {}
for t in (0.3, 0.5, 0.7, 0.9):
    flagged = mx >= t
    tp = int((flagged & has_nodule).sum())
    fn = int((~flagged & has_nodule).sum())
    tn = int((~flagged & ~has_nodule).sum())
    fp = int((flagged & ~has_nodule).sum())
    scan_level[str(t)] = {"accuracy": round((tp + tn) / len(mx), 3), "flagged_with_nodule": tp, "missed": fn,
                          "clean_not_flagged": tn, "clean_flagged": fp}
    print(f"score >= {t}: accuracy {(tp + tn) / len(mx):.2f} | scans with nodules flagged {tp}/{tp + fn} | clean scans left alone {tn}/{tn + fp}")
print("scans with a marked nodule:", int(has_nodule.sum()), "| scans without:", int((~has_nodule).sum()))

summary["scan_level"] = scan_level
json.dump(summary, open(f"{OUT_DIR}/eval_summary.json", "w"), indent=1)
shutil.make_archive("/content/cellIQ_ct_eval", "zip", OUT_DIR)
print("zip updated")